# Per-language GSM8K LoRA fine-tune (Gemma 3 12B, QLoRA)

Trains **one adapter for one language**: set `GSM_LANG` below, then **restart the kernel and run all cells once**.
Repeat for each language (`eo`, `es`, `hmn`, `ig`, `la`, `nl`, `zh`). Restarting between languages gives each run a fresh model and frees GPU memory.

Trains on `train.jsonl` and evaluates on `test.jsonl` from `datasets/gsm8k-<lang>/`.

In [1]:
import os

# ---------------- Settings ----------------
GSM_LANG   = os.environ.get("GSM_LANG", "es")   # change this per run (or set the GSM_LANG env var)

BASE_MODEL = "google/gemma-3-12b-it"
DATA_ROOT  = "../datasets"                      # relative to this notebook (fine_tuning/)
OUT_DIR    = f"gemma3-12b-gsm8k-{GSM_LANG}"

COMPLETION_ONLY   = True   # True = loss only on the answer tokens (not the question)
MAX_LENGTH        = 1024
EVAL_MAX_EXAMPLES = 300    # test examples used for in-training eval; None = full test set (slower)
EVAL_STEPS        = 200
SEED              = 42

# HF_CACHE = "/path/to/big/storage/hf_cache"  # uncomment if home quota is small
# os.environ["HF_HOME"] = HF_CACHE            # must run before importing transformers
print("Language:", GSM_LANG, "| output:", OUT_DIR)

Language: es | output: gemma3-12b-gsm8k-es


## 1. Load the language's train and test sets

In [2]:
from datasets import load_dataset

folder = os.path.join(DATA_ROOT, f"gsm8k-{GSM_LANG}")
train_path = os.path.join(folder, "train.jsonl")
test_path  = os.path.join(folder, "test.jsonl")

for p in (train_path, test_path):
    if not os.path.exists(p):
        raise FileNotFoundError(f"{p} not found. Files in {folder}: {os.listdir(folder) if os.path.isdir(folder) else 'FOLDER MISSING'}")

def prep(ex):
    q = ex["question"].strip()
    a = ex["answer"].strip()
    if COMPLETION_ONLY:
        return {"prompt": [{"role": "user", "content": q}],
                "completion": [{"role": "assistant", "content": a}]}
    return {"messages": [{"role": "user", "content": q},
                         {"role": "assistant", "content": a}]}

def load_split(path):
    d = load_dataset("json", data_files=path, split="train")
    missing = {"question", "answer"} - set(d.column_names)
    if missing:
        raise ValueError(f"{path}: missing columns {missing}; found {d.column_names}")
    n = len(d)
    d = d.filter(lambda ex: bool(ex["question"] and ex["question"].strip() and ex["answer"] and ex["answer"].strip()))
    print(f"{os.path.basename(path)}: {n} rows, {len(d)} usable")
    return d.map(prep, remove_columns=d.column_names)

train_ds = load_split(train_path).shuffle(seed=SEED)
test_ds  = load_split(test_path)

eval_ds = test_ds
if EVAL_MAX_EXAMPLES is not None and len(test_ds) > EVAL_MAX_EXAMPLES:
    eval_ds = test_ds.shuffle(seed=SEED).select(range(EVAL_MAX_EXAMPLES))

print("Train:", len(train_ds), "| Eval used during training:", len(eval_ds))
print("Optimizer steps for 1 epoch (batch 1 x accum 16):", len(train_ds) // 16)

train.jsonl: 255 rows, 255 usable
test.jsonl: 45 rows, 45 usable
Train: 255 | Eval used during training: 45
Optimizer steps for 1 epoch (batch 1 x accum 16): 15


## 2. Tokenizer, template check, and length check

In [3]:
from transformers import AutoTokenizer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def as_messages(ex):
    return ex["prompt"] + ex["completion"] if COMPLETION_ONLY else ex["messages"]

sample = train_ds.select(range(min(3000, len(train_ds))))
lengths = []
for i, ex in enumerate(sample):
    try:
        ids = tokenizer.apply_chat_template(as_messages(ex), tokenize=True, return_dict=True)["input_ids"]
    except Exception as e:
        print("Bad example:", i, e); break
    lengths.append(len(ids))

lengths = np.array(lengths)
print(f"Checked {len(lengths)} examples | median {int(np.median(lengths))} | p95 {int(np.percentile(lengths, 95))} | max {lengths.max()}")
print(f"Over MAX_LENGTH ({MAX_LENGTH}): {(lengths > MAX_LENGTH).mean():.1%}")

Checked 255 examples | median 166 | p95 269 | max 411
Over MAX_LENGTH (1024): 0.0%


## 3. Load the model in 4-bit

In [4]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print("Model loaded")

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

Model loaded


## 4. Train, evaluate, and save

In [5]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

args = SFTConfig(
    output_dir=OUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    num_train_epochs=1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=200,
    save_total_limit=2,
    bf16=True,
    optim="paged_adamw_8bit",
    max_length=MAX_LENGTH,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()

# Save first, so a failure in the evaluation below can never cost you the adapter
FINAL_DIR = os.path.join(OUT_DIR, "final")
trainer.save_model(FINAL_DIR)            # adapter_config.json + adapter_model.safetensors
tokenizer.save_pretrained(FINAL_DIR)
print("Saved adapter to", os.path.abspath(FINAL_DIR))

metrics = trainer.evaluate()
print("Final eval:", metrics)

Tokenizing train dataset:   0%|          | 0/255 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/255 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/255 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/255 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/45 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/45 [00:00<?, ? examples/s]

... (3 outputs left out)

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
16,0.689959,0.601902,0.539397,44325.000000,0.840891


Saved adapter to /workspace/NLP-and-LLMs-Math-by-Language-Project/fine_tuning/gemma3-12b-gsm8k-es/final


Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
0.689959,0.601902,16,0.539397,44325.000000,0.840891


Final eval: {'eval_loss': 0.6019017696380615, 'eval_entropy': 0.5393971112039354, 'eval_num_tokens': 44325.0, 'eval_mean_token_accuracy': 0.8408905016051398}


## Notes
- **Interrupted run:** restart the kernel, re-run sections 0 to 3, then change `trainer.train()` to `trainer.train(resume_from_checkpoint=True)`.
- **Use the adapter:** set `ADAPTER_PATH = "gemma3-12b-gsm8k-<lang>/final"` in `run_finetuned.ipynb`.
- **Run all languages unattended (optional, from a terminal in `fine_tuning/`):**
  ```bash
  for l in eo es hmn ig la nl zh; do
    GSM_LANG=$l jupyter nbconvert --to notebook --execute train_gsm8k_per_language.ipynb \
      --output run_$l.ipynb --ExecutePreprocessor.timeout=-1
  done
  ```
  Each language runs in a fresh process, so GPU memory is released in between.